# Nile-Chat 12B — LoRA Fine-Tuning on Raylab WhatsApp Data

A Colab-ready adaptation of `llm_finetuning.ipynb` — same section order, same
distillation-then-LoRA-then-vLLM shape, retargeted at **Nile-Chat 12B** and the
**490 real, gate-verified, deduplicated Raylab examples** already produced locally
(465 train / 25 val).

**What you will do here:** load the already-prepared dataset from Google Drive,
LoRA fine-tune Nile-Chat 12B via LLaMA-Factory, evaluate before vs. after against
real documented baseline failures, estimate cost/throughput, serve with vLLM, and
load-test.

**What you will NOT do here:** any teacher-distillation or data-formatting work —
that's Stage 0 below, already complete on the source machine.

Runtime: **Colab, 1× A100 GPU** (Runtime → Change runtime type → A100).

## Stage 0 — What's already done (skip these notebook cells)

The reference notebook's cells 8–37 — task definitions, zero-shot baseline check,
cloud-teacher distillation loop, Alpaca formatting — map directly onto work already
completed locally, against real Postgres data, with Claude Opus 5 as the teacher
instead of the notebook's `gpt-4o-mini`. **None of it needs to run in Colab.**

| Notebook section | Your equivalent | Status |
|---|---|---|
| Tasks + zero-shot Evaluation | `golden_outputs_nilechat12b_1.json` | done — 68 real cases reviewed |
| Knowledge Distillation | `scripts/generate_finetuning_dataset.py` | done — 524 raw generated, ~$6.67 spent |
| Format Finetuning Datasets | `finetune_data/{grounding_gate,dedup,format_alpaca}.py` | done — 498 gate-accepted → 490 post-dedup (465 train / 25 val) |

Colab's job starts at **Stage 1** below.

## Stage 1 — Colab environment setup
*(mirrors cells 2–6 of the reference notebook)*

One real difference from the reference notebook: no OpenAI key needed — the
teacher-distillation work is already finished — so that install and login step is dropped.

### Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


### Install

Confirmed from Nile-Chat-12B's real HF model card / `config.json` and
`transformers`' own GitHub release notes: Gemma 3 support needs
**`transformers>=4.50.0`** at minimum (PR #36658).

**LLaMA-Factory's own `check_dependencies()`** (its real source,
`extras/misc.py`) enforces exact ranges on five packages, not just
`transformers` — confirmed live from two real `ImportError`s this notebook hit
in a row (an unbounded `transformers` pin, then a completely unpinned
`datasets`, each resolving past LLaMA-Factory's own ceiling):

| Package | LLaMA-Factory's exact range |
|---|---|
| `transformers` | `transformers>=4.55.0,<=5.8.0,!=4.57.0,!=5.6.0` |
| `datasets` | `datasets>=2.16.0,<=4.0.0` |
| `accelerate` | `accelerate>=1.3.0,<=1.15.0` |
| `peft` | `peft>=0.18.0,<=0.20.0` |
| `trl` | `trl>=0.18.0,<=0.24.0` |

All five are pinned explicitly below — including `peft` and `trl`, which
LLaMA-Factory also requires but which `--no-deps` (see step 5) won't pull in
on its own.

**A second, unrelated issue verified live against PyPI metadata**: `vLLM 0.27.1`
itself hard-pins `torch==2.13.0` + `torchvision==0.28.0` + `torchaudio==2.11.0` as
its own dependencies — but `torch==2.13.0` depends on the CUDA 13.0 toolkit, while
`torchaudio==2.11.0` (still the latest published release) was built against CUDA
12.8. No newer `torchaudio` exists yet that matches, so a plain
`pip install -U torch torchvision torchaudio` on top of Colab's preinstalled torch
reliably reproduces this exact mismatch. The cell below installs everything in the
order vLLM's own docs recommend specifically to avoid it: wipe any preinstalled
torch stack first, then let `uv --torch-backend=auto` resolve a single mutually-
compatible set against Colab's actual CUDA driver, rather than letting each
package resolve to its own independent "latest" from PyPI.

**A third, unrelated issue, verified live against peft's own source**: `peft`'s LoRA adapter setup checks for `torchao` (a quantization library we don't use here) and *raises* `ImportError` if a too-old version is present — confirmed from `peft/import_utils.py`: `is_torchao_available()` returns `False` cleanly when torchao isn't installed at all, but raises when an installed version is below `0.16.0`. `torchao==0.10.0` gets pulled in transitively by the vLLM/torch install above. Since this fine-tune uses no quantization, removing torchao entirely is the safer fix — pinning a new version risks its own CUDA-build mismatch, the same class of problem already hit for torch/torchaudio.

**A fourth addition, added deliberately (not error-driven)**: `bitsandbytes` for the `adamw_bnb_8bit` optimizer used in Stage 5, chosen specifically to avoid lowering `cutoff_len` (see Stage 4/5 below) -- verified real 0.50.1 PyPI metadata confirms CUDA 11.8/12/13 and `torch<3,>=2.4` support, matching the `torch==2.13.0` pinned above.

In [2]:
# 1. Remove Colab's preinstalled torch stack (and anything from a prior manual
#    install) before vLLM brings in its own matched set — mixing sources is what
#    causes the torch/torchaudio CUDA mismatch.
!pip uninstall -y torch torchvision torchaudio vllm

# 2. uv is vLLM's own recommended installer for exactly this problem.
!pip install -qU uv

# 3. --torch-backend=auto detects the real CUDA driver on this Colab A100 runtime
#    and installs one mutually-compatible torch/torchvision/torchaudio set for it,
#    instead of resolving each package independently against PyPI's separate
#    "latest" releases. --system targets Colab's interpreter directly (uv defaults
#    to requiring a virtualenv, which Colab doesn't use).
!uv pip install --system vllm --torch-backend=auto

# 4. torchao (pulled in transitively above) breaks peft's LoRA setup below if an
#    old version is present — peft raises ImportError on torchao<0.16.0, but
#    returns cleanly if it's simply absent. Not needed here (no quantization),
#    so removed rather than upgraded.
!pip uninstall -y torchao

# 5. All five exact ranges LLaMA-Factory's own check_dependencies() enforces
#    (confirmed live from its real source, not guessed one error at a time).
#    transformers' floor here also satisfies the >=4.50.0 Gemma 3 requirement.
!pip install -qU "transformers>=4.55.0,<=5.8.0,!=4.57.0,!=5.6.0" "datasets>=2.16.0,<=4.0.0" "accelerate>=1.3.0,<=1.15.0" "peft>=0.18.0,<=0.20.0" "trl>=0.18.0,<=0.24.0"

# 6. --no-deps: LLaMA-Factory's own setup.py can otherwise pull a loose/unpinned
#    torch requirement and silently re-upgrade it, undoing step 3's matched set.
#    peft/trl are already pinned correctly by step 5 above, so --no-deps here
#    doesn't leave them missing.
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e . --no-deps

# 7. bitsandbytes for the adamw_bnb_8bit optimizer (Stage 5) -- lets us keep
#    cutoff_len at the real full 4096 (zero truncation of any example) by cutting
#    optimizer-state memory instead of dataset context. 0.50.1 (latest, verified on
#    PyPI) declares CUDA 11.8/12/13 + torch<3,>=2.4 support, matching the
#    torch==2.13.0 pinned by step 3 above -- no new CUDA-mismatch risk.
!pip install -qU "bitsandbytes>=0.50.1"

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 103.4 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 196 packages in 1.92s
Prepared 101 packages in 46.28s
Uninstalled 14 packages in 124ms
Installed 101 packages in 265ms
 + anthropic==1.0.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.4
 + compressed-tensors==0.17.0
 - cuda-bindings==12.9.7
 + cuda-bindings==13.3.1
 - cuda-core==0.3.2
 + cuda-core==1.0.1
 - cuda-python==12.9.7
 + cuda-python==13.3.1
 + cuda-tile==1.5.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.2.1
 + depy

### Tokens (optional — only if you want W&B logging or a Hub push)

In [3]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('huggingface')
!huggingface-cli login --token {hf_token}

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mennaharmas316 (mennaharmas316-celltek) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Hint: A new version of huggingface_hub (1.28.0) is available! You are using version 1.27.0.
To update, run: hf update
Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



## Stage 2 — Get your data onto Drive
*(mirrors cells 34–37)*

The reference notebook builds `train.json`/`val.json` in-notebook. Yours already
exist — upload the whole output folder as-is.

1. From your machine, upload the contents of
   `D:\Raylab_Project\scripts\finetune_data_out\` — specifically `train.json`,
   `val.json`, `dataset_info.json` — into a Drive folder, e.g.
   `/gdrive/MyDrive/raylab-finetune/datasets/`.
2. Your existing `dataset_info.json` already uses the correct LLaMA-Factory column
   mapping (`prompt→instruction`, `query→input`, `response→output`, plus
   `system`/`history`) — same shape as the reference notebook's own cell 39. Only the
   `file_name` paths need updating for Drive, done by the cell below.

In [5]:
import json

data_dir = "/gdrive/MyDrive/raylab_finetune/datasets"
info_path = f"{data_dir}/dataset_info.json"

info = json.load(open(info_path, encoding="utf-8"))
info["raylab_finetune_train"]["file_name"] = f"{data_dir}/train.json"
info["raylab_finetune_val"]["file_name"] = f"{data_dir}/val.json"
json.dump(info, open(info_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

!cp {info_path} /content/LLaMA-Factory/data/dataset_info.json
print("dataset_info.json copied into LLaMA-Factory with Drive paths")

dataset_info.json copied into LLaMA-Factory with Drive paths


## Stage 3 — Baseline sanity check
*(mirrors cells 18–22 — recommended, not required)*

You already have a documented baseline (`golden_outputs_nilechat12b_1.json` /
`golden_outputs_nilechat12b_before_fine_tune.json`, 46/67 pass pre-fine-tune). This
step confirms the SAME base model, freshly loaded in Colab, reproduces that
behavior before you spend GPU hours tuning it — reused directly from the real
Helwan-ambulance case, one of the documented fact-inversion failures.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

base_model_id = "MBZUAI-Paris/Nile-Chat-12B"

model = AutoModelForCausalLM.from_pretrained(
    base_model_id, device_map="auto", torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Real, resolved whatsapp_mode_a_reply_directive text (Bucket C) — fetched live via
# TemplateParser against stores/llm/templates/static/system_directives.py, not
# re-typed by hand. This is the exact system prompt Mode A uses in production.
system_prompt = """انت 'سارة'، موظفة خدمة عملاء مصرية ودودة في مركز رايلاب للأشعة والتحاليل الطبية. اتبع القواعد الآتية بالترتيب ده:

1. الدقة في استخراج الحقيقة: شغلانتك الأساسية إنك تجاوب على سؤال المريض بدقة باستخدام المعلومات المكتوبة تحت 'CONTEXT' بس. استخرج الحقيقة اللي بتجاوب على سؤاله، وبعدين اكتبها في جملة طبيعية وودودة بالعامية المصرية — من غير ما تنسخ وتلصق نص الـ CONTEXT حرفيًا زي ما هو. ممنوع تخترع سعر أو اسم أو أي حقيقة مش موجودة قدامك. لو خدمة أو حاجة معينة مكتوب في الـ CONTEXT إنها غير متاحة، قول بوضوح إنها غير متاحة (وأبدًا العكس لو مكتوب إنها متاحة). انقل الأرقام، الأسعار، والمواعيد حرفيًا كما هي مكتوبة في الـ CONTEXT لتجنب أي أخطاء حسابية أو زمنية. إذا سأل المريض عن خدمة طبية، جراحة، أو تخصص (مثل زراعة الأسنان أو الكشف الطبي) غير مذكور ومطابق حرفياً لما هو موجود في الـ CONTEXT، يجب عليك فوراً الاعتذار بلباقة وإخباره أن هذه الخدمة غير متوفرة وأن مركز رايلاب متخصص في الأشعة والتحاليل الطبية فقط. إياك أن تحاول الإجابة باستخدام معلومات عن خدمة أخرى مشابهة، وإياك أن تخترع معلومات من خارج الـ CONTEXT. ممنوع نهائيًا إنك تخترع أو تحسب أي مثال توضيحي بالأرقام من عندك، حتى لو الحساب نفسه صح رياضيًا — المريض ما طلبش الحساب ده، وهو مش مكتوب حرفيًا في الـ CONTEXT. مثال حرفي على اللي ممنوع تمامًا تعمله: لو الـ CONTEXT بيقول 'النسبه: 0.25' والمريض سأل عن نسبة الكاش باك على الأشعة، ❌ ممنوع تضيف جملة زي 'يعني مثلاً لو الأشعة تكلفتها 1000 جنيه، هترجعلك 250 جنيه' — ده مثال مُختلَق من عندك، مش موجود في الـ CONTEXT، حتى لو الحساب نفسه صح. ✅ الرد الصح هو نقل الرقم زي ما هو بس ('نسبة الكاش باك على الأشعة 25% يا فندم')، من غير أي حساب أو مثال إضافي من عندك.

2. المصطلحات الطبية: حافظ على كل المصطلحات الطبية وأسماء الفحوصات (زي MRI، CT، X-Ray، CBC) والأسماء التجارية بالإنجليزي بالظبط زي ما هي مكتوبة في الـ CONTEXT. أي كلمة إنجليزي عامة مش مصطلح طبي (زي 'Services' أو 'Branches') ترجمها للعربي (ممنوع نهائيًا تعريب أو ترجمة أسماء الأشعة والفحوصات، يجب نقلها بالإنجليزي دائمًا كما هي في الـ CONTEXT، حتى لو كان باقي الرد بالعربي).

3. الشخصية والأسلوب: اتكلمي بعامية مصرية طبيعية وصافية 100% — من غير فصحى رسمية جامدة، ومن غير أي لهجة خليجية (ممنوع تمامًا استخدام كلمات خليجية مثل: وش، شلون، أبغى، وايد، أو الفصحى المعقدة). تحدثي بأسلوب الشارع المصري الراقي والودود.

4. أمثلة على الأسلوب المطلوب (جمل كاملة طبيعية، مش كلمات منفصلة لازم تتكرر حرفيًا):
   - الـ CONTEXT بيقول: 'الجمعة مغلق'. المريض: 'مواعيد الجمعة؟' ← الرد: 'يوم الجمعة الفرع بيكون إجازة يا فندم، تحب أحجزلك في يوم تاني؟'
   - الـ CONTEXT بيقول: 'اسانسير: متاح'. المريض: 'فيه أسانسير؟' ← الرد: 'أيوه فيه أسانسير في الفرع يا فندم، تحب تعرف حاجة تانية؟'
   - الـ CONTEXT بيقول: 'فيزا: متاح. فاليو: غير متاح'. المريض: 'بتقبلوا فيزا؟' ← الرد: 'أيوه، الفرع بيقبل فيزا عادي، بس للأسف الفاليو مش متاحة حاليًا. حابب تعرف طريقة دفع تانية؟'

5. سؤال المتابعة: ادمج سؤال المتابعة في نهاية الرد كجملة طبيعية متصلة، وممنوع كتابة أي عناوين وصفية قبله.

6. الأسئلة العامة والواسعة: لو المريض سأل سؤال عام عن الخدمات المتاحة بشكل عام (زي 'عندكم إيه من الأشعة')، اقرأ كل الـ CONTEXT (هيوصلك مقسّم لمصادر مرقمة [BEGIN SOURCE n]...[END SOURCE n]) وطلّع قائمة نقطية بسيطة وواضحة بالعربي للخدمات المتاحة، وخلي كل حقيقة مرتبطة بمصدرها الصح. خليها مختصرة جدًا.

كمان، لو سؤال المريض عن حقيقة محددة (زي مدة تحضير، حد أقصى للوزن، مدة زمنية، أو أي رقم أو شرط معين) — مش سؤال عام عن قائمة خدمات — لكن وصلك أكتر من [BEGIN SOURCE] في نفس الرد:
   - لو أكتر من مصدر بيقول نفس الحقيقة بالظبط (نفس الرقم أو نفس الشرط) لكن كل مصدر مرتبط بفرع مختلف، والمريض ما حددش أي فرع — قول الحقيقة عادي وبثقة من غير ما تسأل عن الفرع أصلاً، لأن الإجابة واحدة في كل الحالات.
   - لو مصدر بيتكلم عن فحص أو خدمة مختلفة تمامًا عن اللي المريض سأل عنها — حتى لو شكله أو تنسيقه (زي جدول أو تصنيف بالأرقام) قريب من اللي محتاجه — تجاهل المصدر ده تمامًا وما تستخدمش أرقامه أو شروطه. حدد المصدر الصح بناءً على إن موضوعه يطابق بالظبط الفحص أو الخدمة اللي المريض سأل عنها، مش مجرد شكل البيانات أو تنسيقها.
   - ممنوع نهائيًا إنك تردي برسالة فاضية أو تكرري سؤال المريض من غير إجابة لمجرد إن قدامك أكتر من مصدر أو قيم متعارضة. لو الحقيقة الصح موجودة في مصدر واحد على الأقل بيتكلم عن نفس اللي اتسأل عنه بالظبط، لازم تقوليها بثقة.
"""

def generate(system, instruction, input_text):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"{instruction}\n{input_text}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    out = out[:, inputs.input_ids.shape[1]:]
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/626 [00:00<?, ?it/s]

In [3]:
# Real Helwan-ambulance case (golden_test_suite.json) — note the expected fact
# itself changed since this suite was first written (source data now says
# ambulance service IS available at Helwan); this cell is checking that the
# Colab-loaded model reproduces the SAME behavior as the documented baseline run,
# not checking correctness in isolation.
print(generate(
    system_prompt,
    "لو حصل طارئ وأنا في فرع حلوان، فيه عربية إسعاف موجودة؟",
    "معلومات الفرع الإسعاف: متاح",
))

لا يا باشا، أنا آسفة جداً، بس احنا مركز أشعة وتحاليل مش مستشفى ولا بنقدم خدمات إسعاف. ممكن تكلم المطافي او الإسعاف مباشرة عشان يساعدوك. ربنا معاك! عايز تعرف اي حاجة تانية؟


> **Free the GPU before the next stage needs it.** The model just loaded here
> (~24GB in bf16 for a 12B model) stays resident in this kernel's GPU memory
> until explicitly freed — it does NOT get released just because the next cell
> doesn't reference it. Stage 5's training subprocess and Stage 8's vLLM server
> each try to load their own full copy on the same GPU; skipping this cleanup
> is exactly what produces a `CUDA out of memory` error whose "already in use"
> figure matches this model's size almost exactly (confirmed live: 23.36 GiB
> in use + 21.92 GiB requested on a 39.49 GiB / 40GB A100, from a real run
> that skipped this step).

In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed")

## Stage 4 — Measure `cutoff_len` for real, don't inherit the notebook's

The reference notebook hardcodes `cutoff_len: 3500` for its own, differently-shaped
dataset. Real measured stats, using Nile-Chat's own tokenizer against the actual
`train.json`/`val.json` (490 records, system + instruction + input + output combined):

| Percentile | Tokens |
|---|---|
| min | 1,506 |
| p50 | 1,549 |
| p90 | 2,136 |
| p95 | 2,451 |
| p99 | 4,045 |
| max | 4,385 |

Two things worth knowing: the fixed system prompt (~1,400+ tokens) means even the
*shortest* real example starts around 1,500 tokens — this dataset doesn't have a lot
of genuinely small examples. And a `cutoff_len` of 4096 (this notebook's original
value) already silently truncates the 3 longest examples, since max is 4,385.

**This measurement is why `cutoff_len` was NOT lowered.** Only 9/490 examples (1.8%) exceed 3,072 tokens -- but all 9 are `broad_positive` (17% of that category), and every one has all 5 retrieved sources contributing real, answer-bearing content. Truncating them would train the model to give a confident, fully-correct-looking multi-source answer from a partially-hidden context -- exactly the ungrounded-confidence pattern this RAG pipeline exists to prevent. `cutoff_len` stays at the notebook's original 4096 (the pre-existing baseline, unchanged by any of this troubleshooting -- note that even 4096 still falls short of the true max of 4,385, so a handful of the very longest examples were already, silently, being truncated before this OOM ever came up; that's an existing, small, unrelated cost, not something introduced by today's decision) and the step-21 OOM is solved with a memory-side fix (`adamw_bnb_8bit`) instead.

Re-run the cell below yourself against your own copy to reproduce these numbers (useful if the dataset changes):

In [ ]:
import json

train = json.load(open(f"{data_dir}/train.json", encoding="utf-8"))
lengths = [
    len(tokenizer.encode(r["system"] + r["instruction"] + r["input"] + r["output"]))
    for r in train
]
print("max real example, tokens:", max(lengths))
print("p99:", sorted(lengths)[int(len(lengths) * 0.99)])

> **Start `cutoff_len` at 4096**, raise it if the measured max above exceeds that.
> Silent truncation here quietly drops exactly the longest, hardest `broad_positive`
> multi-source examples — the ones an entire extra sampling pass was added to get more of.

## Stage 5 — Configure & run LoRA training
*(mirrors cells 39–41)*

Hyperparameters below are reasoned down from the reference notebook's
`lora_rank: 64`, not copied:

- **Rank 16, not 64** — 465 training examples teaching a narrow behavior (grounding +
  JSON discipline), not new knowledge. Lower rank = less capacity to memorize a small
  set verbatim. A 12B base also needs proportionally less adapter capacity to steer
  than the reference notebook's original 1.5B target.
- **`load_best_model_at_end`** — small-N + highly structured JSON output is unusually
  easy to overfit. Selecting by `eval_loss` beats blindly committing to the final
  epoch's checkpoint.

> **Both `model_name_or_path` and `template` are confirmed values** (see Stage 10).
> One correction worth flagging: `template` is `gemma`, not `gemma3` — the `"gemma3"`
> registration wires vision-language support unconditionally (`mm_plugin` with a
> non-None `image_token`), which requires an image processor even for pure-text
> data. Nile-Chat-12B is text-only and ships no processor at all, so `"gemma3"`
> fails on every example with `ValueError: Processor was not found`. The plain
> `"gemma"` template has an identical chat format (verified line-for-line against
> LLaMA-Factory's own source — same `<start_of_turn>`/`<end_of_turn>` tokens) with no
> `mm_plugin` at all, so it's the correct template for a text-only checkpoint, not a
> downgrade.

**Mid-training CUDA OOM — corrected diagnosis after the first fix failed to hold.** A real run first OOM'd at step 21/177. The initial fix (`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`) targeted memory *fragmentation* — but on retry, the exact same run OOM'd at the exact same step 21, with byte-for-byte identical numbers ("tried to allocate 4.19 GiB", "3.97 GiB free", "35.51 GiB in use"). A fragmentation fix would have shifted those numbers; identical reproduction proves this was never fragmentation — it's a genuine peak-memory ceiling, hit deterministically because the same (fixed-seed-shuffled) batch of examples lands at step 21 every run.

Measuring the real token lengths (Stage 4 above) with Nile-Chat's own tokenizer showed only 9/490 examples (1.8%) exceed 3,072 tokens — small in absolute terms, but every one of those 9 is a `broad_positive` example (17% of that category) where all 5 retrieved sources are genuinely answer-bearing. Truncating them would train the model to produce a confident, fully-correct-looking multi-source answer from a partially-hidden context — a real risk of teaching ungrounded confidence, which runs directly against the grounding discipline this whole RAG pipeline is built around. **That trade-off was rejected.** `cutoff_len` stays at the notebook's original 4096 — zero new truncation introduced by this OOM fix.

**Final fix: `optim: adamw_bnb_8bit`** (set in the YAML below), verified live against real sources before adopting — `transformers`' own `OptimizerNames` enum defines `ADAMW_BNB = "adamw_bnb_8bit"`, and `bitsandbytes==0.50.1` (latest, confirmed via PyPI metadata) declares CUDA 11.8/12/13 and `torch<3,>=2.4` support, which covers the `torch==2.13.0` this notebook already pins — no new CUDA-mismatch risk of the kind already hit and fixed for torch/torchaudio earlier. 8-bit AdamW quantizes only the *optimizer's own internal moment estimates* (memory bookkeeping for the training process itself), not the model weights, the LoRA adapters, or any data — it has no path to affect what the model learns from any example, unlike `cutoff_len`, which removes real training signal. This is the textbook use case for an 8-bit optimizer: a memory-side fix for a memory-side problem, leaving the actual training data and target completely untouched. `expandable_segments:True` is left in place on the training invocation (harmless) but isn't relied on as the fix.

In [6]:
%%writefile /content/LLaMA-Factory/examples/train_lora/raylab_finetune.yaml
### model
model_name_or_path: MBZUAI-Paris/Nile-Chat-12B
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
lora_dropout: 0.08
lora_target: all

### dataset
dataset: raylab_finetune_train
eval_dataset: raylab_finetune_val
template: gemma
cutoff_len: 4385  # kept at the real full value -- zero examples truncated. The step-21 OOM
                  # (see Stage 5) is fixed via the adamw_bnb_8bit optimizer below instead,
                  # after truncating the 9 broad_positive examples (17% of that category) was
                  # judged an unacceptable accuracy trade-off, not a memory-only decision
overwrite_cache: true
preprocessing_num_workers: 16

### output
# resume_from_checkpoint: /gdrive/MyDrive/raylab-finetune/models/checkpoint-XXX  # uncomment + fill in after a checkpoint exists, if a run gets interrupted again
output_dir: /gdrive/MyDrive/raylab-finetune/models/
logging_steps: 5
save_steps: 25  # lowered from 50 (still a multiple of eval_steps: 25, required by load_best_model_at_end) after an OOM at step 21 lost all progress
plot_loss: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
optim: adamw_bnb_8bit  # 8-bit AdamW optimizer states -- cuts optimizer memory ~75% vs. fp32
                       # AdamW, freeing enough headroom to keep cutoff_len at the real full
                       # 4096 (see comment above) instead of truncating training data
learning_rate: 1.5e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 25
load_best_model_at_end: true
metric_for_best_model: eval_loss

report_to: wandb
run_name: raylab-nilechat-lora

Writing /content/LLaMA-Factory/examples/train_lora/raylab_finetune.yaml


In [7]:
# PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True: fixes CUDA allocator
# fragmentation from this dataset's wildly varying sequence lengths (short
# narrow_positive vs. long broad_positive examples) — confirmed as the real
# cause from a live OOM traceback ("2.14 GiB reserved but unallocated" at the
# moment of failure, after 21 successful steps, not a first-step OOM). Zero
# accuracy cost — purely a memory-allocator setting, scoped to this one
# subprocess only.
!cd LLaMA-Factory/ && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/raylab_finetune.yaml

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-08-21 12:14:25] llamafactory.hparams.parser:651 >> Process

## Stage 6 — Evaluate: before vs. after
*(mirrors cells 43–49)*

Load base + adapter, reuse the exact same real cases from Stage 3 for a genuine
before/after, not fresh prompts.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 1. تعريف المتغيرات
base_model_id = "MBZUAI-Paris/Nile-Chat-12B"
adapter_dir = "/gdrive/MyDrive/raylab-finetune/models/"

# 2. إعدادات الضغط لـ 8-bit لتوفير المساحة
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# 3. تحميل الموديل الأساسي مضغوط والـ Adapter على الـ GPU
print("Loading base model in 8-bit...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    quantization_config=quantization_config
)

print("Loading adapter...")
model.load_adapter(adapter_dir)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# ... (باقي الكود بتاع الـ system_prompt والـ generate زي ما هو بالظبط) ...

system_prompt = """انت 'سارة'، موظفة خدمة عملاء مصرية ودودة في مركز رايلاب للأشعة والتحاليل الطبية. اتبع القواعد الآتية بالترتيب ده:

1. الدقة في استخراج الحقيقة: شغلانتك الأساسية إنك تجاوب على سؤال المريض بدقة باستخدام المعلومات المكتوبة تحت 'CONTEXT' بس. استخرج الحقيقة اللي بتجاوب على سؤاله، وبعدين اكتبها في جملة طبيعية وودودة بالعامية المصرية — من غير ما تنسخ وتلصق نص الـ CONTEXT حرفيًا زي ما هو. ممنوع تخترع سعر أو اسم أو أي حقيقة مش موجودة قدامك. لو خدمة أو حاجة معينة مكتوب في الـ CONTEXT إنها غير متاحة، قول بوضوح إنها غير متاحة (وأبدًا العكس لو مكتوب إنها متاحة). انقل الأرقام، الأسعار، والمواعيد حرفيًا كما هي مكتوبة في الـ CONTEXT لتجنب أي أخطاء حسابية أو زمنية. إذا سأل المريض عن خدمة طبية، جراحة، أو تخصص (مثل زراعة الأسنان أو الكشف الطبي) غير مذكور ومطابق حرفياً لما هو موجود في الـ CONTEXT، يجب عليك فوراً الاعتذار بلباقة وإخباره أن هذه الخدمة غير متوفرة وأن مركز رايلاب متخصص في الأشعة والتحاليل الطبية فقط. إياك أن تحاول الإجابة باستخدام معلومات عن خدمة أخرى مشابهة، وإياك أن تخترع معلومات من خارج الـ CONTEXT. ممنوع نهائيًا إنك تخترع أو تحسب أي مثال توضيحي بالأرقام من عندك، حتى لو الحساب نفسه صح رياضيًا — المريض ما طلبش الحساب ده، وهو مش مكتوب حرفيًا في الـ CONTEXT. مثال حرفي على اللي ممنوع تمامًا تعمله: لو الـ CONTEXT بيقول 'النسبه: 0.25' والمريض سأل عن نسبة الكاش باك على الأشعة، ❌ ممنوع تضيف جملة زي 'يعني مثلاً لو الأشعة تكلفتها 1000 جنيه، هترجعلك 250 جنيه' — ده مثال مُختلَق من عندك، مش موجود في الـ CONTEXT، حتى لو الحساب نفسه صح. ✅ الرد الصح هو نقل الرقم زي ما هو بس ('نسبة الكاش باك على الأشعة 25% يا فندم')، من غير أي حساب أو مثال إضافي من عندك.

2. المصطلحات الطبية: حافظ على كل المصطلحات الطبية وأسماء الفحوصات (زي MRI، CT، X-Ray، CBC) والأسماء التجارية بالإنجليزي بالظبط زي ما هي مكتوبة في الـ CONTEXT. أي كلمة إنجليزي عامة مش مصطلح طبي (زي 'Services' أو 'Branches') ترجمها للعربي (ممنوع نهائيًا تعريب أو ترجمة أسماء الأشعة والفحوصات، يجب نقلها بالإنجليزي دائمًا كما هي في الـ CONTEXT، حتى لو كان باقي الرد بالعربي).

3. الشخصية والأسلوب: اتكلمي بعامية مصرية طبيعية وصافية 100% — من غير فصحى رسمية جامدة، ومن غير أي لهجة خليجية (ممنوع تمامًا استخدام كلمات خليجية مثل: وش، شلون، أبغى، وايد، أو الفصحى المعقدة). تحدثي بأسلوب الشارع المصري الراقي والودود.

4. أمثلة على الأسلوب المطلوب (جمل كاملة طبيعية، مش كلمات منفصلة لازم تتكرر حرفيًا):
   - الـ CONTEXT بيقول: 'الجمعة مغلق'. المريض: 'مواعيد الجمعة؟' ← الرد: 'يوم الجمعة الفرع بيكون إجازة يا فندم، تحب أحجزلك في يوم تاني؟'
   - الـ CONTEXT بيقول: 'اسانسير: متاح'. المريض: 'فيه أسانسير؟' ← الرد: 'أيوه فيه أسانسير في الفرع يا فندم، تحب تعرف حاجة تانية؟'
   - الـ CONTEXT بيقول: 'فيزا: متاح. فاليو: غير متاح'. المريض: 'بتقبلوا فيزا؟' ← الرد: 'أيوه، الفرع بيقبل فيزا عادي، بس للأسف الفاليو مش متاحة حاليًا. حابب تعرف طريقة دفع تانية؟'

5. سؤال المتابعة: ادمج سؤال المتابعة في نهاية الرد كجملة طبيعية متصلة، وممنوع كتابة أي عناوين وصفية قبله.

6. الأسئلة العامة والواسعة: لو المريض سأل سؤال عام عن الخدمات المتاحة بشكل عام (زي 'عندكم إيه من الأشعة')، اقرأ كل الـ CONTEXT (هيوصلك مقسّم لمصادر مرقمة [BEGIN SOURCE n]...[END SOURCE n]) وطلّع قائمة نقطية بسيطة وواضحة بالعربي للخدمات المتاحة، وخلي كل حقيقة مرتبطة بمصدرها الصح. خليها مختصرة جدًا.

كمان، لو سؤال المريض عن حقيقة محددة (زي مدة تحضير، حد أقصى للوزن، مدة زمنية، أو أي رقم أو شرط معين) — مش سؤال عام عن قائمة خدمات — لكن وصلك أكتر من [BEGIN SOURCE] في نفس الرد:
   - لو أكتر من مصدر بيقول نفس الحقيقة بالظبط (نفس الرقم أو نفس الشرط) لكن كل مصدر مرتبط بفرع مختلف، والمريض ما حددش أي فرع — قول الحقيقة عادي وبثقة من غير ما تسأل عن الفرع أصلاً، لأن الإجابة واحدة في كل الحالات.
   - لو مصدر بيتكلم عن فحص أو خدمة مختلفة تمامًا عن اللي المريض سأل عنها — حتى لو شكله أو تنسيقه (زي جدول أو تصنيف بالأرقام) قريب من اللي محتاجه — تجاهل المصدر ده تمامًا وما تستخدمش أرقامه أو شروطه. حدد المصدر الصح بناءً على إن موضوعه يطابق بالظبط الفحص أو الخدمة اللي المريض سأل عنها، مش مجرد شكل البيانات أو تنسيقها.
   - ممنوع نهائيًا إنك تردي برسالة فاضية أو تكرري سؤال المريض من غير إجابة لمجرد إن قدامك أكتر من مصدر أو قيم متعارضة. لو الحقيقة الصح موجودة في مصدر واحد على الأقل بيتكلم عن نفس اللي اتسأل عنه بالظبط، لازم تقوليها بثقة."""

def generate(system, instruction, input_text):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"{instruction}\n{input_text}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    out = out[:, inputs.input_ids.shape[1]:]
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

import warnings
warnings.filterwarnings('ignore') # السطر ده هيسكت التحذيرات المزعجة تماماً

print("إجابة الموديل بعد التدريب:")
print("-" * 50)
print(generate(
    system_prompt,
    "لو حصل طارئ وأنا في فرع حلوان، فيه عربية إسعاف موجودة؟",
    "معلومات الفرع الإسعاف: متاح"
))
print("-" * 50)

In [3]:
from IPython.display import clear_output

# 1. توليد الإجابة (التحذيرات هتظهر لحظياً هنا)
response = generate(
    system_prompt,
    "لو حصل طارئ وأنا في فرع حلوان، فيه عربية إسعاف موجودة؟",
    "معلومات الفرع الإسعاف: متاح"
)

# 2. مسح الشاشة بالكامل من أي تحذيرات مزعجة
clear_output()

# 3. طباعة النتيجة النظيفة
print("إجابة الموديل بعد التدريب:")
print("-" * 50)
print(response)
print("-" * 50)

إجابة الموديل بعد التدريب:
--------------------------------------------------
```json
{"معلومات الفرع الإسعاف": "متاح"}
```

أيوه يا فندم، الإسعاف متاح عندنا في الفرع، تحب أعرفك تفاصيل تانية عن الفرع؟
--------------------------------------------------


Run this across a spread of cases pulled from the failure taxonomy already
documented against the real pre-fine-tune baseline (46/67 pass, 21 fail):

- At least one **fact inversion** (category A, 9 of 21 baseline fails — the dominant
  category): e.g. anesthesia availability at El-Hawamdeya, Lucky Card age eligibility.
- At least one **numeric fabrication** (category B): e.g. Damietta CT weight limit
  (baseline invented 180kg vs. the real "no limit").
- The **Lucky Card multi-scenario case** — same field label
  (`امكانية عرض لاكي`), five near-identical scenarios, baseline got 4/5 right and
  inverted the 5th. Your strongest real-world `ambiguous_cross_chunk` signal.

Compare each pair's `debug_json` block too, if you route these through the real
`/api/whatsapp/chat` endpoint instead of calling `generate()` directly — that field
is what `scripts/collect_golden_responses.py`'s automated grounding check reads.

## Stage 7 — Cost / throughput estimation
*(mirrors cells 51–53)*

Same methodology as the reference notebook, real patient-query-shaped prompts
instead of Faker-generated Arabic filler, since generic Arabic text doesn't
represent real WhatsApp message length or structure.

In [ ]:
from datetime import datetime
import json

real_queries = [
    r["patient_query"]
    for r in json.load(open("golden_outputs_nilechat12b_1.json", encoding="utf-8"))
]

start = datetime.now()
input_tokens = output_tokens = 0

for q in real_queries[:30]:
    resp = generate(system_prompt, q, "")  # real retrieval context in production
    input_tokens += len(tokenizer.encode(q))
    output_tokens += len(tokenizer.encode(resp))

elapsed = (datetime.now() - start).total_seconds()
print(f"elapsed: {elapsed:.1f}s  tokens/sec: {(input_tokens + output_tokens) / elapsed:.1f}")

> **Free the GPU before the next stage needs it.** The model just loaded here
> (~24GB in bf16 for a 12B model) stays resident in this kernel's GPU memory
> until explicitly freed — it does NOT get released just because the next cell
> doesn't reference it. Stage 5's training subprocess and Stage 8's vLLM server
> each try to load their own full copy on the same GPU; skipping this cleanup
> is exactly what produces a `CUDA out of memory` error whose "already in use"
> figure matches this model's size almost exactly (confirmed live: 23.36 GiB
> in use + 21.92 GiB requested on a 39.49 GiB / 40GB A100, from a real run
> that skipped this step).

In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed")

## Stage 8 — Serve with vLLM
*(mirrors cells 55–59 — matches claude.md §6.4's own deployment target)*

Confirmed: current vLLM (`docs.vllm.ai/en/latest/models/supported_models/`) lists
`Gemma3ForCausalLM` — the exact class Nile-Chat-12B's own `config.json` declares
(`"architectures": ["Gemma3ForCausalLM"]`, `model_type: "gemma3_text"`) — as
supported, **with LoRA support**. An earlier vLLM issue (#14696) flagged real
integration problems, but that was specifically for
`Gemma3ForConditionalGeneration`, the vision-language variant — Nile-Chat-12B is
text-only, so that historical gap doesn't apply here.

In [ ]:
!nohup vllm serve "MBZUAI-Paris/Nile-Chat-12B" \
  --dtype=bfloat16 --gpu-memory-utilization 0.85 \
  --max-lora-rank 16 --enable-lora \
  --lora-modules raylab-lora="/gdrive/MyDrive/raylab-finetune/models/" &

!tail -n 30 nohup.out

In [ ]:
import requests

r = requests.post("http://localhost:8000/v1/completions", json={
    "model": "raylab-lora",
    "prompt": prompt,
    "max_tokens": 400,
    "temperature": 0.3,
})
print(r.json())

## Stage 9 — Load testing
*(mirrors cells 61–65)*

Same substitution as Stage 7 — real patient queries instead of Faker text, so
throughput numbers reflect actual traffic shape.

In [ ]:
%%writefile locust.py
import json, random
from locust import HttpUser, task, between

real_queries = [
    r["patient_query"]
    for r in json.load(open("golden_outputs_nilechat12b_1.json", encoding="utf-8"))
]

class CompletionLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        message = {
            "model": "raylab-lora",
            "prompt": random.choice(real_queries),
            "max_tokens": 400,
            "temperature": 0.3,
        }
        self.client.post("/v1/completions", json=message)

In [ ]:
!locust --headless -f locust.py --host=http://localhost:8000 -u 20 -r 1 -t "60s" --html=locust_results.html

## Stage 10 — Verified values (previously open items)

All four items below were open unknowns when this notebook was first drafted.
Each is now confirmed directly against a primary source and already applied
throughout this notebook — kept here as a single reference, not a to-do list.

| # | Item | Confirmed value | Source |
|---|---|---|---|
| 1 | HF repo id | `MBZUAI-Paris/Nile-Chat-12B` | [model card](https://huggingface.co/MBZUAI-Paris/Nile-Chat-12B) + [config.json](https://huggingface.co/MBZUAI-Paris/Nile-Chat-12B/raw/main/config.json) (`architectures: ["Gemma3ForCausalLM"]`) |
| 2 | LLaMA-Factory `template` | `gemma` (corrected from `gemma3`) | [template.py](https://raw.githubusercontent.com/hiyouga/LLaMA-Factory/main/src/llamafactory/data/template.py) — `"gemma3"` exists as a distinct registration and looked like the obvious match by name, but it unconditionally wires vision-language `mm_plugin` support, which requires an image processor Nile-Chat-12B (text-only) doesn't ship. Confirmed live from a real `ValueError` this notebook hit, not caught by the original source check. `"gemma"` has an identical chat-format registration with no `mm_plugin`. |
| 3 | `transformers` version range | `transformers>=4.55.0,<=5.8.0,!=4.57.0,!=5.6.0` | [v4.50.0 release notes](https://github.com/huggingface/transformers/releases/tag/v4.50.0) (Gemma3 floor, PR #36658) + LLaMA-Factory's own `check_dependencies()` ceiling, confirmed live from a real ImportError this notebook hit |
| 4 | vLLM Gemma 3 support | Yes, incl. LoRA (`Gemma3ForCausalLM`) | [vLLM supported models](https://docs.vllm.ai/en/latest/models/supported_models/) |

One nuance worth keeping in mind: vLLM issue [#14696](https://github.com/vllm-project/vllm/issues/14696)
documents real historical integration problems with Gemma 3 — but specifically for
`Gemma3ForConditionalGeneration` (the vision-language class). Nile-Chat-12B is
confirmed text-only (`Gemma3ForCausalLM`), so that gap doesn't apply to this model.